## 1. Installation and imports

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0"
])

import os
import glob
import random
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from google.colab import files
from tqdm.auto import tqdm

## 2. Basic settings

In [ ]:
seed = 42
image_size = 256
batch_size = 4
num_epochs = 25
learning_rate = 1e-4
threshold = 0.5

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
cv2.setRNGSeed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name())

## 3. Uploading and extracting the dataset

In [ ]:
uploaded = files.upload()
archive_path = next(name for name in uploaded if name.lower().endswith(".zip"))
data_dir = "/content/kidney_data"

with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(data_dir)
del uploaded

for split in ["train", "valid", "test"]:
    images = glob.glob(f"{data_dir}/{split}/images/*.jpg")
    if not images:
        raise ValueError(f"No images found: {data_dir}/{split}/images")
    print(f"{split}: {len(images)} images")

## 4. Reading bounding boxes and creating a mask

In [ ]:
def read_boxes(label_path, image_shape):
    height, width = image_shape
    boxes = []

    with open(label_path) as file:
        for line in file:
            if not line.strip():
                continue

            class_id, xc, yc, bw, bh = map(float, line.split())

            if class_id != 0 or not all(0 <= v <= 1 for v in [xc, yc, bw, bh]) or min(bw, bh) <= 0:
                raise ValueError(f"Incorrect labeling: {label_path}")

            x1 = max(0, int(np.floor((xc - bw / 2) * width)))
            y1 = max(0, int(np.floor((yc - bh / 2) * height)))
            x2 = min(width, int(np.ceil((xc + bw / 2) * width)))
            y2 = min(height, int(np.ceil((yc + bh / 2) * height)))

            if x2 <= x1 or y2 <= y1:
                raise ValueError(f"Empty frame: {label_path}")

            boxes.append((x1, y1, x2, y2))

    return boxes


def box_to_mask(gray, box):
    x1, y1, x2, y2 = box
    height, width = gray.shape
    left, top = max(0, x1 - 12), max(0, y1 - 12)
    right, bottom = min(width, x2 + 12), min(height, y2 + 12)
    crop = gray[top:bottom, left:right]
    padded = cv2.copyMakeBorder(crop, 2, 2, 2, 2, cv2.BORDER_REPLICATE)
    image = cv2.cvtColor(padded, cv2.COLOR_GRAY2BGR)
    rect = (x1 - left + 2, y1 - top + 2, x2 - x1, y2 - y1)
    gc_mask = np.zeros(padded.shape, np.uint8)
    piece = np.zeros((y2 - y1, x2 - x1), np.uint8)

    try:
        cv2.setRNGSeed(seed)
        cv2.grabCut(image, gc_mask, rect, np.zeros((1, 65), np.float64),
                    np.zeros((1, 65), np.float64), 5, cv2.GC_INIT_WITH_RECT)
        local_mask = np.isin(gc_mask, [1, 3]).astype(np.uint8)[2:-2, 2:-2]
        piece = local_mask[y1-top:y2-top, x1-left:x2-left]
    except cv2.error:
        pass

    used_otsu = not piece.any()

    if used_otsu:
        _, piece = cv2.threshold(gray[y1:y2, x1:x2], 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if not piece.any():
        raise ValueError(f"Unable to build a mask for the frame {box}")

    mask = np.zeros(gray.shape, np.uint8)
    mask[y1:y2, x1:x2] = piece

    return mask, used_otsu

## 5. Generating masks for all images


In [ ]:
for split in ["train", "valid", "test"]:
    images_dir = f"{data_dir}/{split}/images"
    labels_dir = f"{data_dir}/{split}/labels"
    masks_dir = f"{data_dir}/{split}/masks"
    os.makedirs(masks_dir, exist_ok=True)
    image_paths = sorted(glob.glob(f"{images_dir}/*.jpg"))
    fallback_count = 0
    box_count = 0

    for image_path in tqdm(image_paths, desc=f"Masks {split}"):
        name = os.path.splitext(os.path.basename(image_path))[0]
        gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        boxes = read_boxes(f"{labels_dir}/{name}.txt", gray.shape)
        combined_mask = np.zeros(gray.shape, np.uint8)

        for box in boxes:
            mask, used_otsu = box_to_mask(gray, box)
            combined_mask = np.maximum(combined_mask, mask)
            fallback_count += int(used_otsu)
            box_count += 1

        cv2.imwrite(f"{masks_dir}/{name}.png", combined_mask * 255)

    print(f"{split}: {len(image_paths)} masks; Otsu has been applied to {fallback_count}/{box_count} objects")

## 6. Inspecting images and generated masks


In [ ]:
image_paths = sorted(glob.glob(f"{data_dir}/train/images/*.jpg"))
selected = random.sample(image_paths, 5)
fig, axes = plt.subplots(5, 3, figsize=(12, 15))

for row, image_path in enumerate(selected):
    name = os.path.splitext(os.path.basename(image_path))[0]
    gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(f"{data_dir}/train/masks/{name}.png", cv2.IMREAD_GRAYSCALE)
    axes[row, 0].imshow(gray, cmap="gray")
    axes[row, 1].imshow(mask, cmap="gray", vmin=0, vmax=255)
    axes[row, 2].imshow(gray, cmap="gray")
    axes[row, 2].imshow(np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.6, vmin=0, vmax=255)

    for col, title in enumerate(["Image", "Pseudo-mask", "Overlay"]):
        axes[row, col].set_title(title)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 7. Dataset and DataLoader

In [ ]:
class KidneyStonesDataset(Dataset):
    def __init__(self, images_dir, masks_dir, augment=False):
        self.images = sorted(glob.glob(f"{images_dir}/*.jpg"))
        self.masks_dir = masks_dir
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_path = self.images[index]
        name = os.path.splitext(os.path.basename(image_path))[0]
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(f"{self.masks_dir}/{name}.png", cv2.IMREAD_GRAYSCALE)

        if image is None or mask is None:
            raise ValueError(f"Unable to read the image or mask: {name}")

        had_foreground = mask.any()
        image = cv2.resize(image, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (image_size, image_size), interpolation=cv2.INTER_NEAREST)

        if had_foreground and not mask.any():
            raise ValueError(f"Mask ({name}) disappeared during downscaling. Increase `image_size` and restart training.")

        if self.augment and random.random() < 0.5:
            image = np.fliplr(image).copy()
            mask = np.fliplr(mask).copy()

        image = image.astype(np.float32) / 127.5 - 1.0
        mask = (mask > 0).astype(np.float32)

        return torch.from_numpy(image).unsqueeze(0), torch.from_numpy(mask).unsqueeze(0)


train_ds = KidneyStonesDataset(f"{data_dir}/train/images", f"{data_dir}/train/masks", augment=True)
train_eval_ds = KidneyStonesDataset(f"{data_dir}/train/images", f"{data_dir}/train/masks")
val_ds = KidneyStonesDataset(f"{data_dir}/valid/images", f"{data_dir}/valid/masks")
test_ds = KidneyStonesDataset(f"{data_dir}/test/images", f"{data_dir}/test/masks")

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
train_eval_loader = DataLoader(train_eval_ds, batch_size=batch_size, shuffle=False, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

images, masks = next(iter(train_loader))
print("Images:", images.shape, "Masks:", masks.shape)

## 8. U-Net and the loss function


In [ ]:
model = smp.Unet(encoder_name="resnet50", encoder_weights="imagenet", in_channels=1, classes=1)
model = model.to(device)

dice_loss_fn = smp.losses.DiceLoss(mode="binary", from_logits=True)
bce_loss_fn = nn.BCEWithLogitsLoss()

def loss_fn(logits, masks):
    return dice_loss_fn(logits, masks) + bce_loss_fn(logits, masks)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

## 9. Metrics and model evaluation

In [ ]:
def calculate_metrics(tp, fp, fn, tn):
    dice = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else 1.0
    iou = tp / (tp + fp + fn) if tp + fp + fn > 0 else 1.0

    return {
        "dice": dice,
        "iou": iou,
        "precision": tp / max(tp + fp, 1),
        "recall": tp / max(tp + fn, 1),
        "specificity": tn / max(tn + fp, 1),
        "accuracy": (tp + tn) / max(tp + fp + fn + tn, 1),
    }


def evaluate(loader):
    model.eval()
    total_loss = 0.0
    tp = fp = fn = tn = 0

    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            logits = model(images)
            total_loss += loss_fn(logits, masks).item() * images.size(0)
            predictions = torch.sigmoid(logits) >= threshold
            targets = masks > 0.5
            tp += (predictions & targets).sum().item()
            fp += (predictions & ~targets).sum().item()
            fn += (~predictions & targets).sum().item()
            tn += (~predictions & ~targets).sum().item()

    metrics = calculate_metrics(tp, fp, fn, tn)
    metrics["loss"] = total_loss / len(loader.dataset)

    return metrics, (tp, fp, fn, tn)

## 10. Training


In [ ]:
history = {"train": [], "valid": []}
learning_rates = []
best_val_loss = float("inf")
best_epoch = 0
model_path = "/content/model.pth"

for epoch in range(1, num_epochs + 1):
    model.train()
    learning_rates.append(optimizer.param_groups[0]["lr"])

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()

    train_metrics, _ = evaluate(train_eval_loader)
    val_metrics, _ = evaluate(val_loader)
    history["train"].append(train_metrics)
    history["valid"].append(val_metrics)
    scheduler.step(val_metrics["loss"])

    print(f"Train loss: {train_metrics['loss']:.4f} | Val loss: {val_metrics['loss']:.4f} | "
          f"Val Dice: {val_metrics['dice']:.4f} | Val IoU: {val_metrics['iou']:.4f}")

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_epoch = epoch
        torch.save(model.state_dict(), model_path)
        print("Best model saved.")

print(f"Best epoch: {best_epoch}, validation loss: {best_val_loss:.4f}")

## 11. Training and validation curves


In [ ]:
epochs = range(1, len(history["train"]) + 1)
metric_names = ["loss", "dice", "iou", "precision", "recall", "specificity", "accuracy"]
fig, axes = plt.subplots(2, 4, figsize=(20, 9))

for ax, metric in zip(axes.flat, metric_names):
    ax.plot(epochs, [result[metric] for result in history["train"]], label="Train")
    ax.plot(epochs, [result[metric] for result in history["valid"]], label="Validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", label="Best epoch")
    ax.set_title("Pixel accuracy (mostly background)" if metric == "accuracy" else metric.capitalize())
    ax.set_xlabel("Epoch")
    if metric != "loss":
        ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3)
    ax.legend()

axes[1, 3].plot(epochs, learning_rates)
axes[1, 3].set_title("Learning rate")
axes[1, 3].set_xlabel("Epoch")
axes[1, 3].set_yscale("log")
axes[1, 3].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Evaluating the best model on the entire test set


In [ ]:
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.eval()

train_result, _ = evaluate(train_eval_loader)
val_result, _ = evaluate(val_loader)
test_result, test_counts = evaluate(test_loader)

for name, result in [("Train", train_result), ("Validation", val_result), ("Test", test_result)]:
    print(name)
    for metric, value in result.items():
        print(f"  {metric}: {value:.4f}")

metrics = ["dice", "iou", "precision", "recall", "specificity", "accuracy"]
x = np.arange(len(metrics))
plt.figure(figsize=(12, 5))
plt.bar(x - 0.25, [train_result[m] for m in metrics], width=0.25, label="Train")
plt.bar(x, [val_result[m] for m in metrics], width=0.25, label="Validation")
plt.bar(x + 0.25, [test_result[m] for m in metrics], width=0.25, label="Test")
plt.xticks(x, metrics)
plt.ylim(0, 1.05)
plt.title("Best model: train / validation / test")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

tp, fp, fn, tn = test_counts
matrix = np.array([[tn, fp], [fn, tp]])
normalized = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
plt.figure(figsize=(6, 5))
plt.imshow(normalized, cmap="Blues", vmin=0, vmax=1)

for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{matrix[i,j]:,}\n({normalized[i,j]:.1%})", ha="center", va="center",
                 bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"})

plt.xticks([0, 1], ["Background", "Stone"])
plt.yticks([0, 1], ["Background", "Stone"])
plt.xlabel("Prediction")
plt.ylabel("Pseudo-mask")
plt.title("Test confusion matrix")
plt.colorbar()
plt.show()

## 13. Test segmentation examples


In [ ]:
indices = random.Random(seed).sample(range(len(test_ds)), 5)
fig, axes = plt.subplots(5, 4, figsize=(16, 18))

for row, index in enumerate(indices):
    image, mask = test_ds[index]

    with torch.no_grad():
        probability = torch.sigmoid(model(image.unsqueeze(0).to(device)))

    prediction = (probability.cpu().numpy()[0, 0] >= threshold)
    gray = (image.numpy()[0] + 1) / 2
    target = mask.numpy()[0]
    axes[row, 0].imshow(gray, cmap="gray")
    axes[row, 1].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[row, 2].imshow(prediction, cmap="gray", vmin=0, vmax=1)
    axes[row, 3].imshow(gray, cmap="gray")
    axes[row, 3].imshow(np.ma.masked_where(~prediction, prediction), cmap="autumn", alpha=0.6, vmin=0, vmax=1)

    for col, title in enumerate(["Image", "Pseudo-mask", "Prediction", "Overlay"]):
        axes[row, col].set_title(title)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()